In [ ]:
%matplotlib inline
from scipy.stats import randint as sp_randint
from scipy.stats import uniform
from scipy.stats import uniform as sp_randFloat
from sklearn import svm
from tqdm import tqdm
from sklearn.naive_bayes import CategoricalNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from time import time
from tabulate import tabulate
import numpy as np
import pandas as pd
import sklearn
import warnings
warnings.filterwarnings('ignore')
import os

In [2]:
from scipy.stats import randint as sp_randInt

from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import make_scorer
from scipy import sparse

In [15]:
import json
with open('GA_output_ET.json', 'r') as fp:
    feature_list = json.load(fp)

In [16]:
feature_list

{'SYN': ['ts',
  'IP_flags',
  'IP_DF',
  'TCP_dataofs',
  'TCP_SYN',
  'TCP_ACK',
  'sport_class',
  'dst_IP_diversity',
  'dst_port_diversity',
  'pck_size_sum_of_EW',
  'ts_diff',
  'ts_std_WE',
  'ts_sum_of_EW',
  'TCP_window_std_WE',
  'pck_size_mean_2',
  'ts_mean_2',
  'ts_std_2',
  'TCP_window_mean_2',
  'TCP_SYN_sum',
  'TCP_ACK_sum',
  'TCP_SYN_ratio',
  'TCP_ACK_SR',
  'ts_mean_6',
  'ts_std_6',
  'pck_size_mean_9',
  'ts_mean_9',
  'ts_std_9',
  'TCP_window_mean_9',
  'TCP_ACK_R',
  'Label'],
 'HTTP': ['ts',
  'TCP_flags',
  'dport_class',
  'pck_size_sum_of_EW',
  'ts_std_WE',
  'ts_sum_of_EW',
  'payload_bytes_sum_of_EW',
  'entropy_mean_WE',
  'TCP_window_mean_2',
  'sport_sum',
  'TCP_ACK_sum',
  'TCP_ACK_ratio',
  'sum',
  'TCP_ACK_SR',
  'ts_std_6',
  'TCP_window_mean_6',
  'TCP_PSH_R',
  'Label'],
 'ACK': ['payload_bytes_mean_WE',
  'sport_sum',
  'TCP_ACK_ratio',
  'TCP_ACK_SR',
  'ts_mean_6',
  'Label'],
 'UDP': ['IP_tos',
  'IP_DF',
  'IP_proto',
  'TCP_dataofs',


In [5]:
def run_random_search(model, param_grid, x_train, y_train, cv=5, scoring='f1_macro'):
    """
    Hàm này thực hiện tìm kiếm ngẫu nhiên (RandomizedSearchCV) để tối ưu hóa siêu tham số của mô hình học máy.

    Tham số:
    - model: mô hình học máy (ví dụ: RandomForestClassifier, SVM, v.v.)
    - param_grid: từ điển chứa các tham số và giá trị cần thử nghiệm (ví dụ: {'n_estimators': [100, 200], 'max_depth': [10, 20]})
    - x_train: Dữ liệu huấn luyện (đặc trưng)
    - y_train: Dữ liệu nhãn tương ứng với x_train
    - cv: số lượng lần chia dữ liệu trong Cross-Validation (mặc định là 5)
    - scoring: phương pháp đánh giá mô hình (mặc định là 'f1_macro')

    Trả về:
    - best_params_: Bộ tham số tốt nhất tìm được
    - best_score_: Điểm số tốt nhất (theo phương pháp đánh giá)
    - best_estimator_: Mô hình học máy với các tham số tốt nhất
    """

    # Thực hiện tìm kiếm ngẫu nhiên với RandomizedSearchCV
    grid = RandomizedSearchCV(model, param_distributions=param_grid, cv=cv, scoring=scoring, n_jobs=-1, verbose=1)
    
    # Huấn luyện mô hình với dữ liệu huấn luyện
    grid.fit(x_train, y_train)
    
    # Trả về các kết quả tốt nhất
    return grid.best_params_, round(grid.best_score_, 8), grid.best_estimator_

In [6]:
def find_the_way(path,file_format,con=""):
    files_add = []
    # r=root, d=directories, f = files
    for r, d, f in os.walk(path):
        for file in f:
            if file_format in file:
                if con in file:
                    files_add.append(os.path.join(r, file))  
            
    return files_add

In [23]:
file_list={
 'SYN': ['./csvs/dos-synflooding-1-dec_SW.csv'],
 'HTTP': ['./csvs/mirai-httpflooding-4-dec_SW.csv'],
}

In [32]:
lines = [['bootst', 'criter', 'max_depth', 'max_features', "min_samp_split", 
          "n_estimators", "F1", "Std", "Time", "No", "Attack"]]

for j in file_list:
    print(f"Processing: {j}")
    
    # Kiểm tra nếu file tồn tại
    if j not in feature_list or len(file_list[j]) < 1:
        print(f"Key {j} không hợp lệ hoặc thiếu file dữ liệu. Bỏ qua...")
        continue

    # Đọc dữ liệu từ file
    df = pd.read_csv(file_list[j][0], usecols=feature_list[j])
    
    # Tách X (feature) và y (label)
    X = df.iloc[:, :-1]
    df['Label'] = df['Label'].astype('category')
    y = df['Label'].cat.codes  

    # Chia dữ liệu thành train và test (80-20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Kiểm tra log
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
    
    second = time()
    clf = RandomForestClassifier(random_state=42)
    
    clf.fit(X_train, y_train)
    predict = clf.predict(X_test)
    f1_score = sklearn.metrics.f1_score(y_test, predict, average="macro")
    
    print(f"F1 Score: {f1_score:.4f}")
    print(f"Thời gian huấn luyện: {round(time() - second, 3)} giây")

    # Ghi lại kết quả với tham số mặc định
    lines.append(["default", "-", "-", "-", "-", "-", f1_score, "-", round(time() - second, 3), "-", j])

    # Tuning tham số với Random Search
    print("Đang thực hiện Random Search...")
    param_dist = {
        "max_depth": sp_randint(1, 32),
        "n_estimators": sp_randint(10, 200),
        "max_features": sp_randint(1, 11),
        "min_samples_split": sp_randint(2, 11),
        "bootstrap": [True, False],
        "criterion": ["gini", "entropy"]
    }

    # Tạo RandomizedSearchCV
    random_search = RandomizedSearchCV(clf, param_distributions=param_dist, n_iter=10, cv=5, verbose=1, random_state=42)
    second = time()
    random_search.fit(X_train, y_train)
    best_clf = random_search.best_estimator_

    # Dự đoán với model tối ưu
    predict = best_clf.predict(X_test)
    f1_score = sklearn.metrics.f1_score(y_test, predict, average="macro")
    
    print(f"F1 Score (Sau Random Search): {f1_score:.4f}")
    print(f"Thời gian Random Search: {round(time() - second, 3)} giây")

    # Ghi lại kết quả sau khi tuning tham số
    best_params = random_search.best_params_
    lines.append([best_params['bootstrap'], best_params['criterion'], best_params['max_depth'],
                  best_params['max_features'], best_params['min_samples_split'], best_params['n_estimators'],
                  f1_score, "-", round(time() - second, 3), "-", j])

# Ghi kết quả vào file CSV
results = pd.DataFrame(lines[1:], columns=lines[0])
results.to_csv("results_RF_tuned.csv", index=False)
print("Kết quả lưu vào file: results_RF_tuned.csv")

print(tabulate(results, headers='keys', tablefmt='psql', showindex=False))

Processing: SYN
Train size: (32630, 29), Test size: (8158, 29)
F1 Score: 0.9999
Thời gian huấn luyện: 0.63 giây
Đang thực hiện Random Search...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
F1 Score (Sau Random Search): 0.9999
Thời gian Random Search: 34.667 giây
Processing: HTTP
Train size: (88306, 17), Test size: (22077, 17)
F1 Score: 0.9989
Thời gian huấn luyện: 4.16 giây
Đang thực hiện Random Search...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
F1 Score (Sau Random Search): 0.9997
Thời gian Random Search: 318.83 giây
Kết quả lưu vào file: results_RF_tuned.csv
+----------+----------+-------------+----------------+------------------+----------------+----------+-------+---------+------+----------+
| bootst   | criter   | max_depth   | max_features   | min_samp_split   | n_estimators   |       F1 | Std   |    Time | No   | Attack   |
|----------+----------+-------------+----------------+------------------+----------------+----------+-------+---------+--

In [34]:
from sklearn.naive_bayes import GaussianNB

In [37]:
lines = [['var_smoothing', 'F1', 'Std', 'Time', 'No', 'Attack']]

for j in file_list:
    print(f"Processing: {j}")
    
    # Kiểm tra nếu file tồn tại
    if j not in feature_list or len(file_list[j]) < 1:
        print(f"Key {j} không hợp lệ hoặc thiếu file dữ liệu. Bỏ qua...")
        continue

    # Đọc dữ liệu từ file
    df = pd.read_csv(file_list[j][0], usecols=feature_list[j])
    
    # Tách X (feature) và y (label)
    X = df.iloc[:, :-1]
    df['Label'] = df['Label'].astype('category')
    y = df['Label'].cat.codes  

    # Chia dữ liệu thành train và test (80-20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Kiểm tra log
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
    
    # Cấu hình tham số tìm kiếm cho Naive Bayes
    param_grid = {
        'var_smoothing': np.logspace(0, -9, num=100),
    }

    # Khởi tạo bộ phân loại Naive Bayes
    second = time()
    clf = GaussianNB()

    # Đào tạo mô hình và đánh giá với F1 Score
    clf.fit(X_train, y_train)
    predict = clf.predict(X_test)
    f1_score = sklearn.metrics.f1_score(y_test, predict, average="macro")
    
    print(f"F1 Score: {f1_score:.4f}")
    print(f"Thời gian huấn luyện: {round(time() - second, 3)} giây")

    # Ghi lại kết quả với tham số mặc định
    lines.append(["default", f1_score, "-", round(time() - second, 3), "-", j])

    # Tuning tham số với RandomizedSearchCV
    print("Đang thực hiện Random Search...")
    
    # Tạo RandomizedSearchCV
    random_search = RandomizedSearchCV(GaussianNB(), param_distributions=param_grid, n_iter=5, cv=5, verbose=1, random_state=42)
    
    for i in range(5):
        second = time()
        random_search.fit(X_train, y_train)
        best_clf = random_search.best_estimator_

        # Dự đoán với model tối ưu
        predict = best_clf.predict(X_test)
        f1_score = sklearn.metrics.f1_score(y_test, predict, average="macro")
        
        print(f"F1 Score (Sau Random Search): {f1_score:.4f}")
        print(f"Thời gian Random Search: {round(time() - second, 3)} giây")

        # Ghi lại kết quả sau khi tuning tham số
        best_params = random_search.best_params_
        lines.append([best_params['var_smoothing'], f1_score, "-", round(time() - second, 3), i, j])

# Ghi kết quả vào file CSV
results = pd.DataFrame(lines[1:], columns=lines[0])
results.to_csv("NB_HPO.csv", index=False)

print("Kết quả lưu vào file: NB_HPO.csv")

# In kết quả cuối cùng
print(tabulate(results, headers='keys', tablefmt='psql', showindex=False))

Processing: SYN
Train size: (32630, 29), Test size: (8158, 29)
F1 Score: 0.9932
Thời gian huấn luyện: 0.026 giây
Đang thực hiện Random Search...
Fitting 5 folds for each of 5 candidates, totalling 25 fits
F1 Score (Sau Random Search): 0.9876
Thời gian Random Search: 0.534 giây
Fitting 5 folds for each of 5 candidates, totalling 25 fits
F1 Score (Sau Random Search): 0.9876
Thời gian Random Search: 0.526 giây
Fitting 5 folds for each of 5 candidates, totalling 25 fits
F1 Score (Sau Random Search): 0.9876
Thời gian Random Search: 0.508 giây
Fitting 5 folds for each of 5 candidates, totalling 25 fits
F1 Score (Sau Random Search): 0.9876
Thời gian Random Search: 0.499 giây
Fitting 5 folds for each of 5 candidates, totalling 25 fits
F1 Score (Sau Random Search): 0.9876
Thời gian Random Search: 0.504 giây
Processing: HTTP
Train size: (88306, 17), Test size: (22077, 17)
F1 Score: 0.6337
Thời gian huấn luyện: 0.035 giây
Đang thực hiện Random Search...
Fitting 5 folds for each of 5 candidates, t